# 02 · Run detectors over a sample

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/02_run_detectors.ipynb)

← [01_datasets](01_datasets.ipynb) · [series index](README.md) · [03_score_and_compare](03_score_and_compare.ipynb) →

Run each detector over every record in the sample and save its predictions.
Detection runs once. Scoring (03), composites (04) and sanitization (05) all
read the saved predictions, so you never re-run a model just to look at the
results differently.

In [ ]:
# Setup — run once per session. On Colab this clones and installs the harness
# (a few minutes the first time). Locally, after `uv sync`, it does nothing.
HARNESS_REPO = "https://github.com/jstjoe/local-privacy.git"
HARNESS_REF = "main"   # branch or tag — e.g. a PR branch to try it before merging
OPF_REPO = "https://github.com/openai/privacy-filter.git"
OPF_REF = "main"

import importlib.util, os, subprocess, sys

if importlib.util.find_spec("opf_eval") is None:
    home = "/content" if "google.colab" in sys.modules else os.path.expanduser("~")
    for repo, ref, dest in [(OPF_REPO, OPF_REF, f"{home}/privacy-filter"),
                            (HARNESS_REPO, HARNESS_REF, f"{home}/local-privacy")]:
        if not os.path.exists(dest):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", ref, repo, dest], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{home}/privacy-filter",
                    f"{home}/local-privacy/eval[notebooks]", f"{home}/local-privacy/api"], check=True)
    # A running kernel doesn't always see freshly pip-installed packages;
    # putting the sources on sys.path makes the imports below work either way.
    sys.path[:0] = [f"{home}/privacy-filter", f"{home}/local-privacy/eval/src", f"{home}/local-privacy/api/src"]

from opf_eval import nb
ws = nb.setup()

## Session

This picks up the dataset and sample chosen in 01. Without one, it uses the
defaults: 100 records of `pii_masking_200k`.

In [ ]:
S = nb.load_session()

## Pick detectors

Local detectors run in this runtime. OPF and the larger GLiNER models are 5–10×
faster on a GPU. **Remote** detectors (Skyflow) call an API and need
credentials.

GLiNER models are *prompted* with label names. The harness prompts them only
with the categories this dataset annotates, so they don't produce labels
nobody will score. `detector_options` overrides per-detector settings, such as
a GLiNER confidence threshold (default 0.7).

In [ ]:
from opf_eval.detectors import registry

nb.show_table(registry.describe(), ["name", "available", "remote", "description"])

In [ ]:
# === EDIT ME ===
S = nb.session(
    detectors=["presidio", "gliner", "gliner_nvidia", "opf"],
    # e.g. {"gliner": {"threshold": 0.5}} — lower = more recall, more false positives
    detector_options={},
)
# ================
S.show()

## Credentials and extra models

Skyflow needs `SKYFLOW_VAULT_URL`, `SKYFLOW_VAULT_ID` and
`SKYFLOW_BEARER_TOKEN`. On Colab, add them under **Secrets** (key icon in the
left sidebar). Locally, set them in your shell or a `.env` file. Presidio
needs a spaCy language model. This cell handles both, and only does what the
chosen detectors need.

In [ ]:
if any(registry.get(d).remote for d in S.detectors):
    found = nb.load_secrets(*nb.SKYFLOW_SECRETS)
    print("Skyflow credentials:", "loaded" if all(found.values()) else f"missing {[k for k, v in found.items() if not v]}")

if any(d.startswith("presidio") for d in S.detectors):
    nb.ensure_spacy_model("en_core_web_lg")

## Run

`ensure_run` materializes the fixtures if needed, then runs every detector
that doesn't have results yet. For each detector the run directory gets:

- `raw_<detector>.jsonl`: one line per record, `{id, spans, latency_ms, error}`
- `manifest.json`: the dataset, annotated labels, a hash of the fixtures, and
  the detectors that ran

A run directory belongs to one fixtures file. Changing the sample in the
session gives a new directory. To re-run a detector with new settings, delete
its `raw_` file or call `runner.run(..., fresh=True)`.

In [ ]:
run_dir = nb.ensure_run(S)
for p in sorted(run_dir.iterdir()):
    print(f"  {p.name}")

## What came back

Here are the predictions for one record from every detector, next to the gold
spans. Look for disagreements on boundaries ("Dr. Jane Smith" vs "Jane
Smith"), labels, and coverage.

In [ ]:
from opf_eval import scoring

run = scoring.Run.load(run_dir)
ex = run.fixtures[0]
print(ex["text"], "\n")

rows = [{"source": "GOLD", "text": ex["text"][s["start"]:s["end"]], "label": s.get("fine_label") or s["label"]}
        for s in ex["gold_spans"]]
for det in run.detectors:
    rec = run.predictions[det][ex["id"]]
    rows += [{"source": det, "text": s["text"], "label": s.get("fine_label") or s["label"]} for s in rec["spans"]]
nb.show_table(rows)

## Speed

These are per-call latencies from the run. Model load time isn't included, and
remote detectors include the network round trip. The first call on a GPU is
slower while kernels warm up, which shows in p99.

In [ ]:
from opf_eval import plots

nb.show_table([{"detector": d, **scoring.latencies(run, d)} for d in run.detectors],
              ["detector", "p50", "p95", "p99", "n"], floatfmt=".1f")
plots.latency_bars(run);

## Next

Score the predictions and compare detectors.

**[03_score_and_compare](03_score_and_compare.ipynb)** — Score, compare, and inspect errors.